# Rogue waves (nonlinear Schrödinger)

*waves · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/nonlinear_schrodinger)

The focusing nonlinear Schrödinger equation describes the envelope of deep-water wave groups and light pulses in fibres. A uniform wave train is unstable: tiny ripples grow into towering peaks about three times the background, like rogue waves. Solved by split-step Fourier: dispersion is exact in $k$-space, the nonlinearity is exact in $x$-space.

**Vector form**

$$i\,\psi_t + \tfrac12\,\psi_{xx} + |\psi|^2\,\psi = 0$$

**Index (tensor) form**

$$\psi_j \leftarrow F^{-1}_{jk}\,e^{-ik^2\Delta t/4}\,F_{kl}\,\psi_l,\qquad \psi_j \leftarrow e^{i\,\psi_j^*\psi_j\,\Delta t}\,\psi_j,\qquad N = \sum_j \psi_j^*\psi_j\,\Delta x$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_j$ | `psi` | (J,) | complex wave envelope on a periodic grid (complex128) |
| $e^{-ik^2\Delta t/4}$ | `half_linear` | (K,) | half a step of dispersion $i\psi_t = -\tfrac12\psi_{xx}$, exact for each Fourier mode |
| $F_{kl}\psi_l$ | `torch.fft.fft(psi)` | (K,) | repeated $l$ ⇒ summed: the discrete Fourier transform |
| $e^{i\vert \psi_j\vert ^2\Delta t}$ | `exp(1j * psi.abs()**2 * dt)` | (J,) | one step of the nonlinearity $i\psi_t = -/\psi/^2\psi$, exact because $/\psi/$ does not change |
| $N,\ H$ | `norm(psi, dx), hamiltonian(psi, k, dx)` | scalar | $\int/\psi/^2$ and $\int \tfrac12/\psi_x/^2 - \tfrac12/\psi/^4$, both conserved |

Strang splitting: half a dispersion step, a full nonlinear step, half a dispersion step; second-order accurate and it keeps $N$ to round-off. Modulational instability: on a background of height 1 perturbations with wavenumber $K < 2$ grow, fastest at $K = \sqrt2$ with rate 1. In **modulation** mode the background starts with $10^{-3}$ random noise (fixed seed): peaks appear after $t \approx 8$ and keep forming and dissolving. The Peregrine breather $\psi = \bigl[1 - \frac{4(1+2it)}{1+4x^2+4t^2}\bigr]e^{it}$ rises out of the background once, reaches exactly $3\times$ its height and vanishes; **peregrine** mode starts it at $t = -5$. **soliton** mode launches the bright soliton $\eta\,\mathrm{sech}(\eta x)\,e^{ivx}$ that keeps its shape forever. The dashed line marks $|\psi| = 3$. Periodic box of length 60.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/nonlinear_schrodinger/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    dt = 0.005
    x, k, dx = tensors.grid(512, dev)  # k_max² Δt / 2 < π avoids the split-step grid instability
    half_linear = tensors.half_linear_step(k, dt)
    psi0 = tensors.initial_wave(x, params["mode"], params["noise"], params["eta"])

    times, psi = solve.solve(psi0, half_linear, dt, params["t_max"])

    N, H = equation.norm(psi, dx), equation.hamiltonian(psi, k, dx)
    print(f"norm        N = {N[0]:.6f}, largest relative change {((N - N[0]).abs().max() / N[0]):.1e}")
    print(f"Hamiltonian H = {H[0]:.6f}, largest change {((H - H[0]).abs().max()):.1e}")
    peak = psi.abs().max(dim=1).values
    when = int(peak.argmax())
    print(f"highest peak |psi| = {peak[when]:.3f} at t = {times[when]:.2f}  (Peregrine breather: 3)")
    return plot.animate(times, x, psi, out_dir / "nonlinear_schrodinger.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/nonlinear_schrodinger/equation.py
r"""Physics → tensor translation.

Focusing nonlinear Schrödinger equation:   i ψ_t + ½ ψ_xx + |ψ|² ψ = 0.

Split it into two parts that can each be solved exactly over a step Δt:

    dispersion   i ψ_t = −½ ψ_xx   →   \hat ψ_k ← e^{−i k² Δt/2} \hat ψ_k              (in Fourier space)
    nonlinearity i ψ_t = −|ψ|² ψ   →   ψ_j ← e^{i ψ_j^* ψ_j Δt} ψ_j                     (in real space, |ψ| fixed)

Strang splitting does half a dispersion step, a full nonlinear step, then another half dispersion step.
In index form the dispersion half step is ψ_j ← F^{-1}_{jk} e^{−ik²Δt/4} F_{kl} ψ_l   (k, l summed).

Conserved: N = ∫ |ψ|² dx and H = ∫ (½ |ψ_x|² − ½ |ψ|⁴) dx.
"""

import cmath

import torch


def split_step(psi: torch.Tensor, half_linear: torch.Tensor, dt: float) -> torch.Tensor:
    """One Strang step.   psi: (J,) complex, half_linear = e^{−ik²Δt/4}: (K,) → (J,)"""
    psi = torch.fft.ifft(half_linear * torch.fft.fft(psi))  # half a step of dispersion
    psi = psi * torch.exp(1j * psi.abs() ** 2 * dt)  # a full step of nonlinearity
    return torch.fft.ifft(half_linear * torch.fft.fft(psi))  # the other half of dispersion


def norm(psi: torch.Tensor, dx: float) -> torch.Tensor:
    """N = Σ_j ψ_j^* ψ_j Δx.   (…, J) → (…)"""
    return (psi.abs() ** 2).sum(dim=-1) * dx


def hamiltonian(psi: torch.Tensor, k: torch.Tensor, dx: float) -> torch.Tensor:
    """H = Σ_j (½ |ψ_x|² − ½ |ψ|⁴) Δx with the spectral derivative ψ_x = F^{-1}[ik F ψ].   (…, J) → (…)"""
    psi_x = torch.fft.ifft(1j * k * torch.fft.fft(psi, dim=-1), dim=-1)
    return (0.5 * psi_x.abs() ** 2 - 0.5 * psi.abs() ** 4).sum(dim=-1) * dx


def peregrine(x: torch.Tensor, t: float) -> torch.Tensor:
    """Exact Peregrine breather ψ = [1 − 4(1 + 2it) / (1 + 4x² + 4t²)] e^{it}, peak |ψ| = 3 at x = t = 0."""
    return (1 - 4 * (1 + 2j * t) / (1 + 4 * x**2 + 4 * t**2)) * cmath.exp(1j * t)


def bright_soliton(x: torch.Tensor, t: float, eta: float, v: float) -> torch.Tensor:
    """Exact soliton ψ = η sech(η(x − vt)) e^{i(vx + (η² − v²) t / 2)}."""
    phase = v * x + 0.5 * (eta**2 - v**2) * t
    return eta / torch.cosh(eta * (x - v * t)) * torch.exp(1j * phase)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/nonlinear_schrodinger/tensors.py
"""Build the concrete tensors: periodic grid, wavenumbers, the dispersion propagator and the starting wave."""

import math

import torch

from .equation import bright_soliton, peregrine

BOX = 60.0
SEED = 0
SOLITON_SPEED = 1.0


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """x (J,) on [−BOX/2, BOX/2), wavenumbers k (K = J,), and Δx."""
    dx = BOX / n
    x = -BOX / 2 + torch.arange(n, device=device, dtype=torch.float64) * dx
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx).to(device=device, dtype=torch.float64)
    return x, k, dx


def half_linear_step(k: torch.Tensor, dt: float) -> torch.Tensor:
    """e^{−i k² Δt / 4}: half a step of ψ̂_t = −i (k²/2) ψ̂.   (K,) complex"""
    return torch.exp(-0.25j * k**2 * dt)


def initial_wave(x: torch.Tensor, mode: str, noise: float, eta: float) -> torch.Tensor:
    """ψ0 (J,) complex128 for the chosen mode."""
    if mode == "peregrine":
        return peregrine(x, -5.0)  # rises to its peak at t = 5
    if mode == "soliton":
        return bright_soliton(x + 10.0, 0.0, eta, SOLITON_SPEED)  # starts at x = −10, moves right
    # modulation: background 1 plus a little complex noise (fixed seed, generated on the CPU so every device agrees)
    gen = torch.Generator().manual_seed(SEED)
    ripples = torch.randn(2, len(x), generator=gen, dtype=torch.float64).to(x.device)
    return 1.0 + noise * (ripples[0] + 1j * ripples[1])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/nonlinear_schrodinger/solve.py
"""Time-march NLS with the split-step Fourier method."""

import torch

from problems._common.progress import steps

from .equation import split_step


def solve(
    psi0: torch.Tensor, half_linear: torch.Tensor, dt: float, t_max: float, n_frames: int = 251
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (F,), ψ (F, J) complex) at n_frames evenly spaced times."""
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // (n_frames - 1))
    psi = psi0
    frames, times = [psi0], [0.0]
    for step in steps(n_steps, "split steps"):
        psi = split_step(psi, half_linear, dt)
        if (step + 1) % save_every == 0:
            frames.append(psi)
            times.append((step + 1) * dt)
    return torch.tensor(times, dtype=torch.float64, device=psi0.device), torch.stack(frames)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/nonlinear_schrodinger/plot.py
"""Top: |ψ(x)| now, with the Peregrine level 3. Bottom: the x–t map of |ψ| filling in as time runs."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, x: torch.Tensor, psi: torch.Tensor, path: Path) -> Path:
    t, xs = times.cpu().numpy(), x.cpu().numpy()
    amp = psi.abs().cpu().numpy()  # (F, J)
    running_max = np.maximum.accumulate(amp.max(axis=1))  # (F,)
    top = max(3.3, 1.1 * amp.max())

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7, 4.8), gridspec_kw={"height_ratios": [1, 1.3]}, layout="constrained"
    )
    ax1.axhline(3.0, ls="--", color="#f59e0b", lw=1)
    line = ax1.plot(xs, amp[0], color="#2563eb", lw=1.5)[0]
    info = ax1.text(0.01, 0.95, "", transform=ax1.transAxes, va="top", fontsize=9)
    ax1.set_xlim(xs[0], xs[-1])
    ax1.set_ylim(0, top)
    ax1.set_ylabel("|ψ|")
    title = ax1.set_title("focusing NLS   t = 0.0")

    shown = np.full_like(amp, np.nan)
    image = ax2.imshow(
        shown, origin="lower", aspect="auto", extent=(xs[0], xs[-1], t[0], t[-1]), cmap=plt.get_cmap("magma_r", 12),
        vmin=0, vmax=top, interpolation="nearest",
    )  # fmt: skip
    ax2.set_xlabel("x")
    ax2.set_ylabel("t")
    ax2.grid(False)
    fig.colorbar(image, ax=[ax1, ax2], label="|ψ|", pad=0.01, shrink=0.6, anchor=(0, 0))  # both panels keep one x-axis

    def update(step: int):
        line.set_ydata(amp[step])
        shown[: step + 1] = amp[: step + 1]
        image.set_data(shown)
        info.set_text(f"highest |ψ| so far: {running_max[step]:.2f}")
        title.set_text(f"focusing NLS   t = {t[step]:.1f}")

    return save_gif(fig, update, frame_indices(len(t), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mode': 'modulation',
    'noise': 0.001,
    'eta': 1.5,
    't_max': 25.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.nonlinear_schrodinger")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.nonlinear_schrodinger.main")
problem_dir = Path("problems/waves/nonlinear_schrodinger")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
